In [34]:
import pandas as pd
import os 
import sys
import glob
sys.path.append(os.path.abspath(os.path.join('..')))
from config import DATASETS,ML_MODELS
import joblib
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score
from sklearn.preprocessing import OrdinalEncoder,LabelEncoder
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

latest csv file


In [104]:
def latest_file():
    file_path = glob.glob(os.path.join(DATASETS,"*.csv"))
    if not file_path:
        return None
    else:
        latest_file = max(file_path,key = os.path.getmtime)
        df = pd.read_csv(latest_file)
        return df

In [105]:
df = latest_file()
df['Churn'].isna().sum()

np.int64(0)

split into x and y

In [106]:
x = df.drop(columns='Churn')

le = LabelEncoder()
y = le.fit_transform(df["Churn"])
y

array([0, 0, 1, ..., 0, 1, 0], shape=(7043,))

Train test split

In [107]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
x_test

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
437,4376-KFVRS,Male,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),114.05,8468.2
2280,2754-SDJRD,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.15,908.55
2235,9917-KWRBE,Female,0,Yes,Yes,41,Yes,Yes,DSL,Yes,Yes,Yes,No,Yes,No,One year,Yes,Credit card (automatic),78.35,3211.2
4460,0365-GXEZS,Male,0,Yes,No,18,Yes,No,Fiber optic,No,No,Yes,Yes,No,No,Month-to-month,No,Electronic check,78.20,1468.75
3761,9385-NXKDA,Female,0,Yes,No,72,Yes,Yes,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),82.65,5919.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5143,5204-HMGYF,Female,0,Yes,Yes,49,Yes,No,DSL,Yes,Yes,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,87.20,4345
4439,9950-MTGYX,Male,0,Yes,Yes,28,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card (automatic),20.30,487.95
3857,3675-EQOZA,Male,0,No,No,5,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),20.65,93.55
4758,3646-ITDGM,Female,0,No,No,56,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.70,1051.9


load clean pipeline and clean data

In [108]:
# load pipeline
file_path = os.path.join(ML_MODELS,'Data_cleaning_Pipeline.pkl')
preprocessing = joblib.load(file_path)

In [109]:
# clean data
x_train_pre = preprocessing.transform(x_train)
x_train_clean = pd.DataFrame(x_train_pre,columns=preprocessing.get_feature_names_out())
x_test_pre = preprocessing.transform(x_test)
x_test_clean = pd.DataFrame(x_test_pre,columns=preprocessing.get_feature_names_out())
x_test_clean


,totalcharge__TotalCharges,binary__gender,binary__Partner,binary__Dependents,binary__PhoneService,binary__OnlineSecurity,binary__OnlineBackup,binary__DeviceProtection,binary__TechSupport,binary__StreamingTV,...,nominal__Contract_Month-to-month,nominal__Contract_One year,nominal__Contract_Two year,nominal__PaymentMethod_Bank transfer (automatic),nominal__PaymentMethod_Credit card (automatic),nominal__PaymentMethod_Electronic check,nominal__PaymentMethod_Mailed check,fixed__SeniorCitizen,numeric_col__tenure,numeric_col__MonthlyCharges
0,2.731119,1.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.613701,1.638143
1,-0.606314,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,2.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.992402,1.176164
2,0.410260,0.0,1.0,1.0,1.0,2.0,2.0,2.0,0.0,2.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.351370,0.451620
3,-0.358997,1.0,1.0,0.0,1.0,0.0,0.0,2.0,2.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.585198,0.446635
4,1.605853,0.0,1.0,0.0,1.0,2.0,2.0,2.0,0.0,2.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.613701,0.594535
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1404,0.910809,0.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.677133,0.745758
1405,-0.792000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,-0.177995,-1.477726
1406,-0.966120,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.114563,-1.466094
1407,-0.543028,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.962175,-1.497668


Machine learning models

In [41]:
lr = LogisticRegression(penalty='l2',max_iter=1000,solver='lbfgs',class_weight='balanced')
lr.fit(x_train_clean,y_train)

c:\Users\hamen\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [42]:
lr_test_pr = lr.predict(x_test_clean)
lr_train_pr = lr.predict(x_train_clean)
lr_test_pr

array([0, 1, 0, ..., 0, 0, 0], shape=(1409,))

In [43]:
print(classification_report(y_test,lr_test_pr))
print(classification_report(y_train,lr_train_pr))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

              precision    recall  f1-score   support

           0       0.91      0.73      0.81      4139
           1       0.52      0.81      0.63      1495

    accuracy                           0.75      5634
   macro avg       0.72      0.77      0.72      5634
weighted avg       0.81      0.75      0.77      5634



In [44]:
print(confusion_matrix(y_test,lr_test_pr))

[[747 288]
 [ 80 294]]


In [45]:
cross_val_score(lr,x_train_clean,y_train).mean()

c:\Users\hamen\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\hamen\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\hamen\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was depr

np.float64(0.7520353789828541)

Random forest 

In [46]:
rf = RandomForestClassifier(n_estimators=5000,max_depth=10,min_samples_split=4,max_features='sqrt',class_weight='balanced',random_state=60)
rf.fit(x_train_clean,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",5000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",4
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

In [47]:
rf_test_pr = rf.predict(x_test_clean)
rf_train_pr = rf.predict(x_train_clean)
rf_test_pr

array([0, 1, 0, ..., 0, 0, 0], shape=(1409,))

In [48]:
print(classification_report(y_test,rf_test_pr))
print(classification_report(y_train,rf_train_pr))

              precision    recall  f1-score   support

           0       0.88      0.79      0.84      1035
           1       0.55      0.71      0.62       374

    accuracy                           0.77      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.79      0.77      0.78      1409

              precision    recall  f1-score   support

           0       0.96      0.85      0.90      4139
           1       0.69      0.91      0.78      1495

    accuracy                           0.87      5634
   macro avg       0.82      0.88      0.84      5634
weighted avg       0.89      0.87      0.87      5634



In [49]:
print(confusion_matrix(y_test,rf_test_pr))

[[821 214]
 [110 264]]


In [51]:
cross_val_score(rf,x_train_clean,y_train,cv=5).mean()

KeyboardInterrupt: 

xgboost

In [52]:
# Define model
xb = XGBClassifier(
    n_estimators=5000,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.4,
    colsample_bytree=0.8,
    random_state=50
)

In [53]:
xb.fit(x_train_clean,y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [54]:
y_test_pred = xb.predict(x_test_clean)
y_train_pred = xb.predict(x_train_clean)
print(classification_report(y_test,y_test_pred))
print(classification_report(y_train,y_train_pred))

              precision    recall  f1-score   support

           0       0.84      0.87      0.85      1035
           1       0.60      0.53      0.57       374

    accuracy                           0.78      1409
   macro avg       0.72      0.70      0.71      1409
weighted avg       0.78      0.78      0.78      1409

              precision    recall  f1-score   support

           0       0.93      0.96      0.94      4139
           1       0.87      0.80      0.83      1495

    accuracy                           0.92      5634
   macro avg       0.90      0.88      0.89      5634
weighted avg       0.91      0.92      0.91      5634



In [55]:
print(confusion_matrix(y_test,y_test_pred))

[[902 133]
 [174 200]]


catboost

In [91]:
# define catboost classifier model
catboost = CatBoostClassifier(iterations=5000,
learning_rate=0.05,
depth=2,
l2_leaf_reg =3,
random_state=42)

In [92]:
catboost.fit(x_train_clean,y_train)

0:	learn: 0.6652241	total: 4.28ms	remaining: 21.4s
1:	learn: 0.6394576	total: 7.86ms	remaining: 19.6s
2:	learn: 0.6175039	total: 12.1ms	remaining: 20.1s
3:	learn: 0.5977897	total: 16.1ms	remaining: 20.2s
4:	learn: 0.5840276	total: 20ms	remaining: 20s
5:	learn: 0.5691022	total: 24ms	remaining: 19.9s
6:	learn: 0.5574548	total: 27.7ms	remaining: 19.7s
7:	learn: 0.5469606	total: 31.2ms	remaining: 19.4s
8:	learn: 0.5353283	total: 34.5ms	remaining: 19.1s
9:	learn: 0.5264349	total: 38.2ms	remaining: 19s
10:	learn: 0.5186289	total: 41.9ms	remaining: 19s
11:	learn: 0.5103733	total: 45.9ms	remaining: 19.1s
12:	learn: 0.5026793	total: 51.2ms	remaining: 19.6s
13:	learn: 0.4972920	total: 55.8ms	remaining: 19.9s
14:	learn: 0.4920029	total: 59.6ms	remaining: 19.8s
15:	learn: 0.4875491	total: 63.8ms	remaining: 19.9s
16:	learn: 0.4823434	total: 68.1ms	remaining: 20s
17:	learn: 0.4767894	total: 71.9ms	remaining: 19.9s
18:	learn: 0.4731950	total: 75.8ms	remaining: 19.9s
19:	learn: 0.4689031	total: 79.6ms

CatBoostClassifier(depth=2, iterations=5000, l2_leaf_reg=3, learning_rate=0.05, random_state=42)

In [93]:
ct_test_pred = catboost.predict(x_test_clean)
ct_train_pred = catboost.predict(x_train_clean)

In [94]:
ct_train_pred

array([0, 0, 0, ..., 1, 0, 0], shape=(5634,))

In [95]:
print("Test classification Report : \n",classification_report(y_test,ct_test_pred))
print("Train classification Report : \n",classification_report(y_train,ct_train_pred))

Test classification Report : 
               precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409

Train classification Report : 
               precision    recall  f1-score   support

           0       0.87      0.93      0.90      4139
           1       0.75      0.61      0.67      1495

    accuracy                           0.84      5634
   macro avg       0.81      0.77      0.79      5634
weighted avg       0.84      0.84      0.84      5634



In [96]:
print(confusion_matrix(y_test,ct_test_pred))

[[932 103]
 [176 198]]


In [61]:
cross_val_score(catboost,x_train_clean,y_train,cv=3).mean()

0:	learn: 0.6651778	total: 3.02ms	remaining: 15.1s
1:	learn: 0.6388757	total: 6.93ms	remaining: 17.3s
2:	learn: 0.6189279	total: 10.6ms	remaining: 17.7s
3:	learn: 0.6001162	total: 14.5ms	remaining: 18.1s
4:	learn: 0.5833010	total: 17.3ms	remaining: 17.3s
5:	learn: 0.5692126	total: 20.5ms	remaining: 17.1s
6:	learn: 0.5547862	total: 23.4ms	remaining: 16.7s
7:	learn: 0.5422597	total: 26.3ms	remaining: 16.4s
8:	learn: 0.5335743	total: 30.6ms	remaining: 17s
9:	learn: 0.5257651	total: 34.3ms	remaining: 17.1s
10:	learn: 0.5188140	total: 37.8ms	remaining: 17.2s
11:	learn: 0.5102052	total: 41.4ms	remaining: 17.2s
12:	learn: 0.5048482	total: 44.7ms	remaining: 17.1s
13:	learn: 0.4990382	total: 47.6ms	remaining: 17s
14:	learn: 0.4933291	total: 50.9ms	remaining: 16.9s
15:	learn: 0.4888583	total: 54ms	remaining: 16.8s
16:	learn: 0.4854420	total: 57.2ms	remaining: 16.8s
17:	learn: 0.4817983	total: 61.1ms	remaining: 16.9s
18:	learn: 0.4783694	total: 64.4ms	remaining: 16.9s
19:	learn: 0.4749595	total: 

np.float64(0.7880724174653887)

In [62]:
# on the basis of all accuracy and confusion matrix i find that catboost give more accuary and balance model accuracy in train and test data

create pipeline

In [111]:
full_Pipeline = Pipeline([
    ("clean",preprocessing),
    ('model',CatBoostClassifier(iterations=5000,learning_rate=0.05,depth=2,l2_leaf_reg =3,random_state=42))
])

In [114]:
x_train_clean.columns.tolist()

['totalcharge__TotalCharges',
 'binary__gender',
 'binary__Partner',
 'binary__Dependents',
 'binary__PhoneService',
 'binary__OnlineSecurity',
 'binary__OnlineBackup',
 'binary__DeviceProtection',
 'binary__TechSupport',
 'binary__StreamingTV',
 'binary__StreamingMovies',
 'binary__PaperlessBilling',
 'nominal__MultipleLines_No',
 'nominal__MultipleLines_No phone service',
 'nominal__MultipleLines_Yes',
 'nominal__InternetService_DSL',
 'nominal__InternetService_Fiber optic',
 'nominal__InternetService_No',
 'nominal__Contract_Month-to-month',
 'nominal__Contract_One year',
 'nominal__Contract_Two year',
 'nominal__PaymentMethod_Bank transfer (automatic)',
 'nominal__PaymentMethod_Credit card (automatic)',
 'nominal__PaymentMethod_Electronic check',
 'nominal__PaymentMethod_Mailed check',
 'fixed__SeniorCitizen',
 'numeric_col__tenure',
 'numeric_col__MonthlyCharges']

In [115]:
x_train.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges']

In [116]:
full_Pipeline.fit(x_train,y_train)

0:	learn: 0.6652241	total: 5.78ms	remaining: 28.9s
1:	learn: 0.6394576	total: 11.1ms	remaining: 27.6s
2:	learn: 0.6175039	total: 15.4ms	remaining: 25.7s
3:	learn: 0.5977897	total: 19.8ms	remaining: 24.7s
4:	learn: 0.5840276	total: 24ms	remaining: 24s
5:	learn: 0.5691022	total: 28.6ms	remaining: 23.8s
6:	learn: 0.5574548	total: 33.1ms	remaining: 23.6s
7:	learn: 0.5469606	total: 37.3ms	remaining: 23.3s
8:	learn: 0.5353283	total: 43.2ms	remaining: 23.9s
9:	learn: 0.5264349	total: 47.8ms	remaining: 23.9s
10:	learn: 0.5186289	total: 52.7ms	remaining: 23.9s
11:	learn: 0.5103733	total: 57ms	remaining: 23.7s
12:	learn: 0.5026793	total: 61.4ms	remaining: 23.5s
13:	learn: 0.4972920	total: 65.6ms	remaining: 23.4s
14:	learn: 0.4920029	total: 69.5ms	remaining: 23.1s
15:	learn: 0.4875491	total: 73.8ms	remaining: 23s
16:	learn: 0.4823709	total: 77.9ms	remaining: 22.8s
17:	learn: 0.4783674	total: 82ms	remaining: 22.7s
18:	learn: 0.4747092	total: 86.1ms	remaining: 22.6s
19:	learn: 0.4716130	total: 90ms

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('clean', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('totalcharge', ...), ('binary', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

save file model

In [121]:
file_name_create = f"Customer_churn_ML_model.pkl"
folder_path = os.path.join(ML_MODELS,file_name_create)
folder_path

'c:\\Users\\hamen\\OneDrive\\Desktop\\ml_project\\Telco_Customer_Churn_ML_Project\\ML_Pipelines\\Customer_churn_ML_model.pkl'

In [122]:
joblib.dump(full_Pipeline,folder_path)


['c:\\Users\\hamen\\OneDrive\\Desktop\\ml_project\\Telco_Customer_Churn_ML_Project\\ML_Pipelines\\Customer_churn_ML_model.pkl']